<p><font size="6" color='grey'> <b>
Machine Learning
</b></font> </br></p>


<p><font size="5" color='grey'> <b>
Supervised Learning - Random Forest - Diamonds
</b></font> </br></p>

---


# 0  | Install & Import
***

In [ ]:
# Install

In [ ]:
# Daten & Strukturen
from pandas import read_csv, DataFrame  # CSV laden und Daten verwalten

# Datenvorverarbeitung & Feature Engineering
from sklearn.preprocessing import OrdinalEncoder, StandardScaler  # Encoding und Skalierung
from sklearn.decomposition import PCA  # Dimensionsreduktion

# Datenaufteilung
from sklearn.model_selection import train_test_split  # Train/Test-Split

# Modell
from sklearn.ensemble import RandomForestRegressor  # Ensemble-Regressionsmodell

# Evaluation & Metriken
from sklearn.metrics import r2_score, mean_absolute_error  # Regressionsmetriken

# Modellanalyse & Visualisierung
from yellowbrick.regressor import residuals_plot  # Residuenanalyse
from yellowbrick.regressor import prediction_error  # Prognosefehler
from yellowbrick.model_selection import feature_importances  # Feature-Wichtigkeit

# Visualisierung
import plotly.express as px  # Interaktive Plots
import plotly.subplots as sp  # Subplots erstellen

# Persistenz
import joblib  # Modelle speichern und laden

In [ ]:
# Warnung ausstellen
import warnings
warnings.filterwarnings("ignore")

# 1  | Understand
***

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Aufgabe verstehen</br>
✅ Daten sammeln</br>
✅ Statistische Analyse (Min, Max, Mean, Korrelation, ...)</br>
✅ Datenvisualisierung (Streudiagramm, Box-Plot, ...)</br>
✅ Prepare Schritte festlegen</br>

<p><font color='black' size="5">
Anwendungsfall
</font></p>

---

Dieser klassische Datensatz enthält die Preise und andere Attribute von fast 54.000 Diamanten.



[DataSet](https://www.openml.org/search?type=data&status=active&id=42225)

[Info](https://www.kaggle.com/datasets/shivam2503/diamonds)


In [ ]:
df = read_csv(
    "https://raw.githubusercontent.com/ralf-42/ML_Intro/main/02_daten/05_tabellen/diamonds.csv",
    usecols=[
        "carat",
        "cut",
        "color",
        "clarity",
        "depth",
        "table",
        "price",
    ],
)

In [ ]:
data = df.copy()
target = data.pop("price")

# 2 |  Prepare

---

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Datentyp ermitteln/ändern</br>
✅ Train-Test-Split durchführen</br>
✅ Nicht benötigte Features löschen</br>
✅ Missing Values behandeln</br>
✅ Ausreißer behandeln</br>
✅ Kategorischer Features Kodieren</br>
✅ Numerischer Features skalieren</br>
✅ Feature-Engineering (neue Features schaffen)</br>
✅ Dimensionalität reduzieren</br>
✅ Resampling (Over-/Undersampling)</br>
✅ Pipeline erstellen/konfigurieren</br>

<p><font color='black' size="5">
Datentyp ermitteln
</font></p>

In [ ]:
all_col = data.columns
num_col = data.select_dtypes(include="number").columns
cat_col = data.select_dtypes(exclude="number").columns

<p><font color='black' size="5">
Train-Test-Set
</font></p>


In [ ]:
data_train, data_test, target_train, target_test = train_test_split(
    data, target, test_size=0.2, random_state=42)

<p><font color='black' size="5">
Kodierung
</font></p>

In [ ]:
cat_seq = [
    ["Fair", "Good", "Very Good", "Premium", "Ideal"],
    ["J", "I", "H", "G", "F", "E", "D"],
    ["I1", "SI2", "SI1", "VS2", "VS1", "VVS2", "VVS1", "IF"],]

encoder = OrdinalEncoder(categories=cat_seq)
encoder.fit(data_train[cat_col])
data_train[cat_col] = encoder.transform(data_train[cat_col])
data_test[cat_col] = encoder.transform(data_test[cat_col])

<font color='black' size="5">
Skalierung
</font>

In [ ]:
scaler = StandardScaler()
scaler.fit(data_train[all_col])
data_train[all_col] = scaler.transform(data_train[all_col])
data_test[all_col] = scaler.transform(data_test[all_col])

# 3 |  Modeling
---

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Modellauswahl treffen</br>
✅ Pipeline erweitern/konfigurieren</br>
✅ Training durchführen</br>
✅ Hyperparameter Tuning</br>
✅ Cross-Valdiation</br>
✅ Bootstrapping</br>
✅ Regularization</br>

<p><font color='black' size="5">🌲 Algorithmus: Random Forest (Zufälliger Wald – Regression)</font></p>

Ein Random Forest besteht aus vielen einzelnen Entscheidungsbäumen. Jeder Baum wird auf einer zufälligen Teilmenge der Trainingsdaten und Merkmale trainiert. Bei der Regression ist die finale Vorhersage der Mittelwert aller Baum-Vorhersagen – das macht das Modell robuster als ein einzelner Baum.

**Analogie:** Statt einen einzelnen Experten zu befragen, werden viele unabhängige Experten befragt – der Durchschnitt ihrer Schätzungen ist zuverlässiger.

**Wichtige Parameter:**

| Parameter | Bedeutung |
|-----------|----------|
| `n_estimators` | Anzahl der Bäume im Wald |
| `max_depth` | Maximale Tiefe jedes Baums |
| `max_features` | Zufällig gewählte Merkmale je Split |
| `random_state` | Reproduzierbarkeit |

In [ ]:
#@markdown   <p><font size="4" color='green'>  Random Forest – Regression</font> </br></p>

import base64
from IPython.display import Image, display

diagram = """
flowchart TD
    IN[Eingabe: Diamant-Daten] --> B1[Baum 1]
    IN --> B2[Baum 2]
    IN --> B3[Baum n...]

    subgraph Wald [Random Forest Ensemble]
    B1 --> P1[Vorhersage: $850]
    B2 --> P2[Vorhersage: $720]
    B3 --> P3[Vorhersage: $780]
    end

    P1 --> AGG{Aggregation}
    P2 --> AGG
    P3 --> AGG

    AGG --> OUT[Finaler Preis: $783]

    style Wald fill:#f9f,stroke:#333,stroke-width:2px
"""

encoded = base64.urlsafe_b64encode(diagram.strip().encode()).decode()
display(Image(url=f"https://mermaid.ink/img/{encoded}", width=500))

 <p><font color='black' size="5">
Modellauswahl & Training
</font></p>

In [ ]:
model = RandomForestRegressor()

In [ ]:
model.fit(data_train, target_train)

# 4 | Evaluate
---

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Prognose (Train, Test) erstellen</br>
✅ Modellgüte prüfen</br>
✅ Residuenanalyse erstellen</br>
✅ Feature Importance/Selektion prüfen</br>
✅ Robustheitstest erstellen</br>
✅ Modellinterpretation erstellen</br>
✅ Sensitivitätsanalyse erstellen</br>
✅ Kommunikation (Key Takeaways)</br>


<p><font color='black' size="5">
Prediction
</font></p>


In [ ]:
target_train_pred = model.predict(data_train)
target_test_pred = model.predict(data_test)

<p><font color='black' size="5">
Bestimmtheitsmass
</font></p>

In [ ]:
r2 = r2_score(target_train, target_train_pred)
print(f"Modell: {model} -- Train --- Bestimmtheitsmass: {r2:5.2f}")

In [ ]:
r2 = r2_score(target_test, target_test_pred)
print(f"Modell: {model} -- Test --- Bestimmtheitsmass: {r2:5.2f}")

<p><font color='black' size="5">
Mean Absolut Error
</font></p>

In [ ]:
mae = mean_absolute_error(target_test, target_test_pred)
print(f"Modell: {model} -- Test -- Mean Absolute Error: {mae:5.2f}")

<p><font color='black' size="5">
Aufbau Analysewürfel
</font></p>

In [ ]:
# Übernahme der Testdaten
cube = data_test.copy()
cube.reset_index(inplace=True)

# Übernahme Target real & predict
cube["real"] = DataFrame(target_test.values, columns=["real"])
cube["predict"] = DataFrame(target_test_pred, columns=["predict"])

In [ ]:
# Erstellung 2D Features über Dimensionsreduktion PCA - unsupervised
pca = PCA(n_components=2)
pca = pca.fit_transform(data_test)
pca_df = DataFrame(pca)

# Cube um pca erweitern
cube["PCA1"] = pca_df[0]
cube["PCA2"] = pca_df[1]

<p><font color='black' size="5">
Visualisierung real vs predict
</font></p>

In [ ]:
# Boxplot
title_ = "Boxplot real vs predict"
px.box(cube[["real", "predict"]], title=title_, width=600, height=600)

In [ ]:
# Histogramm
title_ = "Histogramm Prices real vs predict"
fig = px.histogram(cube, x=["real", "predict"], nbins=10, title=title_)
fig.update_layout(barmode="group", bargap=0.2, width=800, height=600)
fig.show()

In [ ]:
# 2 x Scatterplots
title_ = "Streupunktdiagramm real"
img1 = px.scatter(
    cube, x="PCA1", y="PCA2", color="real", title=title_, width=600, height=600
)

title_ = "Streupunktdiagramm predict"
img2 = px.scatter(
    cube, x="PCA1", y="PCA2", color="predict", title=title_, width=600, height=600
)

fig = sp.make_subplots(
    rows=1, cols=2, subplot_titles=("Scatterplot real", "Scatterplot predict")
)

for trace in img1.data:
    fig.add_trace(trace, row=1, col=1)

for trace in img2.data:
    fig.add_trace(trace, row=1, col=2)

# Layout anpassen
fig.update_layout(width=1000, height=500, title_text="Vergleich real vs predict")

# Plot anzeigen
fig.show()

<p><font color='black' size="5">
Fehlerhafte Vorhersagen
</font></p>

In [ ]:
cube["abs_Abw%"] = abs((cube["real"] - cube["predict"]) / cube["real"] * 100)
%precision 3
cube.head(10).style.format("{:,.1f}", subset=num_col)

In [ ]:
cube.describe().T

In [ ]:
# Histogramm
title_ = "Histogramm absolute Abweichung"
fig = px.histogram(cube, x=["abs_Abw%"], nbins=20, title=title_)
fig.update_layout(barmode="group", bargap=0.2, width=800, height=600)
fig.show()

In [ ]:
_ = residuals_plot(
    model,
    data_train,
    target_train,
    data_test,
    target_test,
    train_color="b",
    test_color="g",
)

In [ ]:
_ = prediction_error(model, data_train, target_train, data_test, target_test)


<p><font color='black' size="5">
Feature Importance
</font></p>

In [ ]:
px.bar(
    x=model.feature_importances_, y=data.columns, width=1000, height=600
).update_yaxes(categoryorder="total ascending")

In [ ]:
_ = feature_importances(model, data, target)

# 5 | Deploy
---

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Modellexport und -speicherung</br>
✅ Abhängigkeiten und Umgebung</br>
✅ Sicherheit und Datenschutz</br>
✅ In die Produktion integrieren</br>
✅ Tests und Validierung</br>
✅ Dokumentation & Wartung</br>

<p><font size="5">
Save
</p>

In [ ]:
# schnellste Variante
joblib.dump(model, "/content/diamonds-model.pkl")

<p><font color='black' size="5">
Load
</font></p>

In [ ]:
model_geladen = joblib.load("/content/diamonds-model.pkl")

<p><font size="5">
Prognose
</p>

In [ ]:
target_pred = model_geladen.predict(data_test)
r2 = r2_score(target_test, target_pred)
print(f"Bestimmheitsmaß Testdaten: {r2:.2f}")

In [ ]:
result = DataFrame()
result["real"] = target_test.values
result["pred"] = target_pred
result["%delta"] = (result["real"] - result["pred"]) / result["real"] * 100